# Aula 13 - Revisão para a Prova de 02/10: autodiagnóstico

**Módulo 03 IN** - Lógica para predição com inteligência artificial
**30/09/2026 - Sprint 5 - Prof. Ovidio Lopes da Cruz Netto**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/canaldoovidio/2026-2A-M03/blob/main/notebooks/aula13.ipynb)

## Como usar este notebook

Este é o estudo prévio da revisão, feito **antes** do encontro de 30/09, junto com o
[material de apoio da Aula 13](https://canaldoovidio.github.io/2026-2A-M03/materiais/aula13.html).
São dezesseis questões sobre o case da LDC, organizadas pelos temas de computação da Prova. Cada
uma pede um número calculado sobre os CSVs de `dados/` ou uma letra.

1. Escreva a resposta na variável `resposta_qN`, no lugar do `None`.
2. Rode a célula. A função `verificar` confere a resposta contra um resumo criptográfico: o
   gabarito não está escrito em lugar nenhum do notebook.
3. Resposta certa imprime `correto`. Resposta errada levanta `AssertionError` com uma dica.

Os objetos daqui são de propósito **diferentes** dos que o deck mostra: o deck usa a regressão
logística e a reta de frangos sobre suínos, e este notebook pede o KNN, a baseline sazonal, o
Naive Bayes e a reta de frangos sobre leite. Anote as duas questões em que mais errou: elas vão
para o quadro na daily.

A seção 9 é a prática aberta do encontro, sem verificação automática: ela é corrigida por outra
dupla, em sala.

## 1. As bases e a função de verificação

A célula abaixo monta a base mensal das Aulas 07 a 12 (339 linhas, 315 de treino e 24 de teste,
de 2024-04 a 2026-03) e a base trimestral das cinco séries. Nada aqui é novo.

In [ ]:
import calendar
import hashlib
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import confusion_matrix, r2_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SERIES = ["abate_bovinos", "abate_suinos", "abate_frangos",
          "producao_ovos", "producao_leite"]
ALVO = "abate_frangos"
FEATURES = (["lag1", "lag2", "lag3", "lag12", "sen", "cos", "dias"]
            + [s + "_lag1" for s in SERIES if s != ALVO])
N_TESTE = 24
BRUTA = "https://raw.githubusercontent.com/canaldoovidio/2026-2A-M03/main/dados/"


def caminho(relativo):
    """Usa o CSV do repositório; no Colab, baixa a cópia bruta do GitHub."""
    local = os.path.join("..", "dados", relativo)
    if os.path.exists(local):
        return local
    arquivo = relativo.replace("/", "_")
    if not os.path.exists(arquivo):
        try:
            urllib.request.urlretrieve(BRUTA + relativo, arquivo)
        except Exception as erro:
            raise RuntimeError(
                "Nao foi possivel baixar '%s' (%s). Se a rede da sala falhou, peca a pasta "
                "'dados' para uma dupla que tenha o repositorio clonado e coloque-a ao lado "
                "deste notebook." % (relativo, erro)) from erro
    return arquivo


def serie(relativo, nome):
    return (pd.read_csv(caminho(relativo))[["periodo", "valor"]]
            .rename(columns={"valor": nome}))


# base mensal
base = None
for nome in SERIES:
    coluna = serie("mensal/%s.csv" % nome, nome)
    base = coluna if base is None else base.merge(coluna, on="periodo", how="inner")
base = base.sort_values("periodo").reset_index(drop=True)
base["mes"] = base["periodo"].str[-2:].astype(int)
base["dias"] = [calendar.monthrange(int(p[:4]), int(p[-2:]))[1] for p in base["periodo"]]
base["sen"] = np.sin(2 * np.pi * base["mes"] / 12)
base["cos"] = np.cos(2 * np.pi * base["mes"] / 12)
for k in (1, 2, 3, 12):
    base["lag%d" % k] = base[ALVO].shift(k)
for nome in SERIES:
    if nome != ALVO:
        base[nome + "_lag1"] = base[nome].shift(1)
base = base.dropna().reset_index(drop=True)
corte = len(base) - N_TESTE
X_tr, y_tr = base[FEATURES].to_numpy()[:corte], base[ALVO].to_numpy()[:corte]
X_te, y_te = base[FEATURES].to_numpy()[corte:], base[ALVO].to_numpy()[corte:]

# base trimestral
trim = None
for nome in SERIES:
    coluna = serie("%s.csv" % nome, nome)
    trim = coluna if trim is None else trim.merge(coluna, on="periodo", how="inner")
trim = trim.sort_values("periodo").reset_index(drop=True)


def mape(real, previsto):
    return float(np.mean(np.abs((real - previsto) / real)) * 100)


print("base mensal    : %d linhas, teste de %s a %s"
      % (len(base), base["periodo"].iloc[corte], base["periodo"].iloc[-1]))
print("base trimestral: %d linhas, de %s a %s"
      % (len(trim), trim["periodo"].iloc[0], trim["periodo"].iloc[-1]))

In [ ]:
# Resumo criptográfico de cada resposta, formato esperado e dica.
# O gabarito em si não está escrito aqui: só o sha256 da resposta normalizada.
GABARITO = {
    "q1": ("3df70b3cc026897956f222dfea826998a71fe6e48eee3dc1df47c8dc95f15f62", 'letras', 'Pergunte, coluna a coluna: ela já foi publicada quando a previsão do mês t é feita?'),
    "q2": ("6f38ca849ac4bec96fa656e6e26c0b7e130245b7d3312d542c73a25c6796d608", 0, 'Conte com np.unique sobre as 24 previsões de teste.'),
    "q3": ("4be2b26a28a0aecdca6ab16564d1ac2df7ec47dc1edd3f3157a628e353e87db7", 2, 'Pipeline com StandardScaler e KNeighborsRegressor(n_neighbors=5), fit só nos 315 meses.'),
    "q4": ("8478011234e81e9ba8f2b24502971ac0f2248969ef6d0ea38bf60a6967f80160", 2, 'Mesmo Pipeline, fit nas 339 linhas, medido nos 24 meses finais.'),
    "q5": ("00fa76789a65eac7e36c85148f2573eb431661284e5e522569ba7283701f58e7", 1, 'Erro = previsto - real, com previsto = lag12. MAE em milhões de kg.'),
    "q6": ("ba4cad59a0f2f2ac399bf427d7443ee5a7e9ae0483d03176fd33da74e8b6bf7e", 1, 'Viés é a média do erro com sinal, previsto - real, em milhões de kg.'),
    "q7": ("23b991e62141dfcbde43915782ede22f098b23eb6b58275e34c905f9547fda8f", 'letras', 'Releia o sinal do número da q6.'),
    "q8": ("65cf3a686bf28e3cf4ace0d1404929d26ad1f3455353f9848f31c00b4b7d4f51", 2, 'np.corrcoef sobre as 116 linhas, leite do trimestre anterior contra frangos.'),
    "q9": ("ceb490ece3fb75de1b2a8f9175277ff072cd24b1c7b75fcc7fb954c094d15e26", 2, 'train_test_split(X, y, test_size=0.2, random_state=42) e r2_score no teste.'),
    "q10": ("723a8b76ed5977546140caff28e2fb7d696c5ba764825cb6c63c75eb63b4157b", 2, 'Treine com as 92 primeiras linhas e meça o R2 nas 24 últimas, sem sortear.'),
    "q11": ("b02ccee3436efff15b0239bf9dd64227578bf7777d510a3c6385ff934af622f7", 'letras', 'Confira a unidade da coluna de leite no CSV.'),
    "q12": ("56cb89922581d946cf2e6611afc2764a2c5b5f3aa43b1f7ecd3432e75bb0c5e4", 'inteiros', 'confusion_matrix(real, previsto, labels=[0, 1]).ravel() devolve tn, fp, fn, tp.'),
    "q13": ("9151a88c22174948efbd61e0685a2a07d341e58efbc252846a110b179c917677", 2, 'Revocação da alta = TP / (TP + FN), com previsto = probabilidade >= 0,8.'),
    "q14": ("bb9aed5bba01bf96dbd3c6016798d0d25dfbe22803a849d7ae7b5c50ee3ea149", 'letras', 'Qual dos dois erros ficou caro neste enunciado?'),
    "q15": ("0d78d7b713773532a48e1d0a1b08c632b52e4cfdac1c4c789dddb955282257a9", 'letras', 'O esquema existe antes da carga, ou depois?'),
    "q16": ("5d1e27ddf84980efd7bbbbe1d346bf554623a38bcbbd488fa79889618d4409e8", 'letras', 'Quem nunca foi comprado só pode ser descrito pelos próprios atributos.'),
}
RESULTADO = {}


def _normalizar(formato, resposta):
    if formato == "letras":
        return "".join(sorted(set(str(resposta).upper().replace(",", "").replace(" ", ""))))
    if formato == "inteiros":
        return ",".join(str(int(v)) for v in resposta)
    texto = format(float(resposta), ".%df" % formato)
    if float(texto) == 0:
        texto = format(0.0, ".%df" % formato)
    return texto


def verificar(chave, resposta):
    """Confere a resposta. Sem resposta, só avisa; resposta errada levanta AssertionError."""
    resumo, formato, dica = GABARITO[chave]
    if resposta is None:
        print("%s: sem resposta ainda." % chave)
        return
    normal = _normalizar(formato, resposta)
    certo = hashlib.sha256(("%s|%s" % (chave, normal)).encode()).hexdigest() == resumo
    RESULTADO[chave] = certo
    assert certo, "%s: ainda não. Dica: %s" % (chave, dica)
    print("%s: correto." % chave)

## 2. Variáveis preditoras, alvo e o momento de cada dado

O objetivo do Modelo 1 é prever o abate de frangos do mês t. A pergunta que decide X é quando cada
coluna passa a existir. A base supõe que o mês t-1 já foi publicado no momento da previsão.

**q1.** Quais destas colunas podem entrar em X para prever o abate de frangos do mês t? Responda
com as letras juntas, por exemplo `"AB"`.

- A) `lag12`, o abate de frangos do mesmo mês do ano anterior
- B) `abate_suinos` do próprio mês t
- C) `cos`, o cosseno do mês
- D) `producao_leite_lag1`, o leite do mês anterior
- E) `abate_frangos` do próprio mês t

In [ ]:
resposta_q1 = None
verificar("q1", resposta_q1)

**q2.** Crie a coluna `linha = np.arange(len(base))` e treine uma
`RandomForestRegressor(n_estimators=300, random_state=42)` usando **só essa coluna**, com os 315
meses de treino. Quantos valores distintos a floresta prevê nos 24 meses de teste?

In [ ]:
# escreva aqui o seu código
resposta_q2 = None
verificar("q2", resposta_q2)

## 3. Treino, teste e a validade da estimativa de erro

**q3.** Monte `Pipeline([("escala", StandardScaler()), ("modelo", KNeighborsRegressor(n_neighbors=5))])`,
ajuste com `X_tr, y_tr` e meça o MAPE (em %, duas casas) nos 24 meses de teste.

**q4.** Agora ajuste o mesmo `Pipeline` nas 339 linhas e meça o MAPE nos mesmos 24 meses. Compare
com a q3 e escreva, numa frase, por que o número da q4 não estima o erro em 2026.

In [ ]:
# escreva aqui o seu código
resposta_q3 = None
resposta_q4 = None
verificar("q3", resposta_q3)
verificar("q4", resposta_q4)

## 4. Métricas de regressão

A baseline sazonal ingênua prevê, para cada mês, o valor do mesmo mês do ano anterior: previsto =
`lag12`. Use erro = previsto - real, nos 24 meses de teste, em **milhões de kg**, uma casa.

**q5.** Qual o MAE da baseline sazonal?

**q6.** Qual o viés (erro médio com sinal)?

**q7.** O que o sinal do viés diz? A) que ela erra pouco; B) que ela prevê, em média, abaixo do
real; C) que o MAE está negativo; D) que ela superestima o abate.

In [ ]:
# escreva aqui o seu código
resposta_q5 = None
resposta_q6 = None
resposta_q7 = None
verificar("q5", resposta_q5)
verificar("q6", resposta_q6)
verificar("q7", resposta_q7)

## 5. O fluxo completo de uma regressão linear simples

Base trimestral. y = abate de frangos do trimestre; x = produção de leite do **trimestre
anterior** (`trim["producao_leite"].shift(1)`), que existe antes do trimestre previsto. Depois do
`shift`, descarte a linha vazia: ficam 116. Lembre: X precisa ser 2D (`df[["coluna"]]`) e y 1D
(`df["coluna"]`). A célula abaixo desenha a dispersão, que é o primeiro passo para checar a forma
linear.

In [ ]:
leite = trim[["periodo", "abate_frangos", "producao_leite"]].copy()
leite["leite_lag1"] = leite["producao_leite"].shift(1)
leite = leite.dropna().reset_index(drop=True)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(leite["leite_lag1"] / 1e6, leite["abate_frangos"] / 1e9, s=14)
ax.set_xlabel("leite no trimestre anterior (bilhões de litros)")
ax.set_ylabel("frangos (bilhões de kg)")
plt.show()

**q8.** Qual o r de Pearson entre `leite_lag1` e `abate_frangos`, nas 116 linhas? Duas casas.

**q9.** Com `train_test_split(X, y, test_size=0.2, random_state=42)`, qual o R² de teste da
`LinearRegression`? Duas casas.

**q10.** Sem sortear: treine nas 92 primeiras linhas e meça o R² nas 24 últimas. Duas casas.

**q11.** A inclinação da reta sorteada vale cerca de 560,5. Qual leitura está certa? A) cada litro
de leite a mais acompanha 560,5 kg de frango; B) cada mil litros de leite a mais no trimestre
anterior acompanham, em média, 560,5 kg a mais de frango; C) captar mais leite faz subir o abate
de frango; D) cada mil litros acompanham 560,5 milhões de kg de frango.

In [ ]:
# escreva aqui o seu código
resposta_q8 = None
resposta_q9 = None
resposta_q10 = None
resposta_q11 = None
verificar("q8", resposta_q8)
verificar("q9", resposta_q9)
verificar("q10", resposta_q10)
verificar("q11", resposta_q11)

## 6. Classificação, matriz de confusão e limiar

O alvo binário da Aula 09: alta (1) quando o abate do mês supera o mesmo mês do ano anterior. A
célula abaixo monta o alvo e padroniza X com o escalador ajustado só no treino.

In [ ]:
alvo = (base[ALVO] > base["lag12"]).astype(int).to_numpy()
a_tr, a_te = alvo[:corte], alvo[corte:]
escala = StandardScaler().fit(X_tr)
Z_tr, Z_te = escala.transform(X_tr), escala.transform(X_te)

**q12.** Treine `GaussianNB()` em `Z_tr, a_tr` e preveja `Z_te`. Responda a matriz como a tupla
`(TP, FN, FP, TN)`, com alta como classe positiva.

**q13.** Treine `LogisticRegression(max_iter=2000, random_state=42)`, pegue
`predict_proba(Z_te)[:, 1]` e classifique como alta quando a probabilidade for pelo menos 0,8.
Qual a revocação da classe alta? Duas casas.

**q14.** Suponha que, para a LDC, estoque parado de milho (FP: previu alta e veio queda) custe
muito mais que comprar tarde (FN). O que fazer com o limiar de 0,5? A) baixar para 0,4; B) mantê-lo;
C) subir para 0,7; D) escolher o de maior acurácia.

In [ ]:
# escreva aqui o seu código
resposta_q12 = None
resposta_q13 = None
resposta_q14 = None
verificar("q12", resposta_q12)
verificar("q13", resposta_q13)
verificar("q14", resposta_q14)

## 7. Tipos de dado, ETL e ELT, *data lake* e *data warehouse*

**q15.** A LDC define um esquema fixo com período, série e valor. Cada carga do SIDRA é limpa e
convertida antes de entrar, e os painéis consultam em segundos. O que é isso? A) *data warehouse*
alimentado por ETL; B) *data lake* alimentado por ELT; C) *data warehouse* alimentado por ELT;
D) *data lake* alimentado por ETL.

In [ ]:
resposta_q15 = None
verificar("q15", resposta_q15)

## 8. Sistemas de recomendação e *cold start*

**q16.** A LDC lança uma formulação de ração nova, que nenhuma granja comprou ainda. Qual
estratégia consegue recomendá-la? A) colaborativa usuário-usuário; B) colaborativa item-item;
C) popularidade; D) baseada em conteúdo.

In [ ]:
resposta_q16 = None
verificar("q16", resposta_q16)

### Placar

A célula abaixo mostra quantas questões você respondeu certo nesta sessão. As que ficaram de fora
ou erradas são as que vão para o quadro na daily.

In [ ]:
certas = sorted(k for k, v in RESULTADO.items() if v)
print("respondidas certo: %d de %d" % (len(certas), len(GABARITO)))
print("faltam:", ", ".join(k for k in GABARITO if k not in certas) or "nenhuma")

## 9. Prática aberta do encontro: o fluxo completo sobre o abate de bovinos

Sem verificação automática. A folha da dupla vai para a dupla vizinha às 11h40, e a correção
segue o critério do slide 35 do deck.

1. y = abate de bovinos do trimestre. Escolha **uma** feature que exista antes do trimestre
   previsto e escreva quando ela passa a existir.
2. Confira qualidade: vazios, período coberto e unidade.
3. Escolha a divisão e a proporção, e justifique para 116 trimestres.
4. Treine, reporte MAE, RMSE e R² na unidade certa, e interprete a inclinação.
5. Preveja 2026-T2 e escreva o que a métrica garante, e o que não garante, sobre esse número.

In [ ]:
bov = trim[["periodo", "abate_bovinos"]].copy()
# escreva aqui o fluxo da dupla